## News Category Classification Model

In [1]:
# generate hugginface token or login to hf account
import huggingface_hub
huggingface_hub.login()

In [3]:
# import the necessary libraries and modules
try:
    import datasets, evaluate, accelerate
    import gradio as gr
except ModuleNotFoundError:
    !pip install datasets evaluate accelerate gradio
    import datasets, evaluate, accelerate

import torch
import transformers
import random
import numpy as np
import pandas as pd

# see version of the libraries
print(f"transformers version: {transformers.__version__}")
print(f"datasets version: {datasets.__version__}")
print(f"evaluate version: {evaluate.__version__}")
print(f"accelerate version: {accelerate.__version__}")
print(f"gradio version: {gr.__version__}")


transformers version: 5.15.1
datasets version: 4.0.0
evaluate version: 0.4.6
accelerate version: 1.14.0
gradio version: 6.26.0


### Load Dataset

In [4]:
from datasets import load_dataset

# load the dataset
dataset = load_dataset(path="AiresPucrs/News-Category-Dataset")
dataset

README.md:   0%|          | 0.00/847 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 27.5MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/209527 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 209527
    })
})

In [10]:
print(dataset.column_names)
print(dataset['train'].features)
print(dataset['train'][0])

{'train': ['text', 'labels']}
{'text': Value('string'), 'labels': Value('string')}
{'text': 'Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.', 'labels': 'U.S. NEWS'}


In [13]:
# unique labels in the dataset
labels = dataset['train'].unique("labels")
print(f"Unique labels: {labels}")
print(f"Number of unique labels: {len(labels)}")

Unique labels: ['U.S. NEWS', 'COMEDY', 'PARENTING', 'WORLD NEWS', 'CULTURE & ARTS', 'TECH', 'SPORTS', 'ENTERTAINMENT', 'POLITICS', 'WEIRD NEWS', 'ENVIRONMENT', 'EDUCATION', 'CRIME', 'SCIENCE', 'WELLNESS', 'BUSINESS', 'STYLE & BEAUTY', 'FOOD & DRINK', 'MEDIA', 'QUEER VOICES', 'HOME & LIVING', 'WOMEN', 'BLACK VOICES', 'TRAVEL', 'MONEY', 'RELIGION', 'LATINO VOICES', 'IMPACT', 'WEDDINGS', 'COLLEGE', 'PARENTS', 'ARTS & CULTURE', 'STYLE', 'GREEN', 'TASTE', 'HEALTHY LIVING', 'THE WORLDPOST', 'GOOD NEWS', 'WORLDPOST', 'FIFTY', 'ARTS', 'DIVORCE']
Number of unique labels: 42


In [14]:
# turn the dataset into a pandas dataframe and check some samples
news_df = pd.DataFrame(dataset['train'])
news_df.sample(5)

,text,labels
195837,Jerry Sandusky Guilty: Verdict Reached In Chil...,CRIME
180109,5 Excellent Honeymoon Destinations You Don't N...,WEDDINGS
61368,Don't Play Pokemon Go During State Department ...,POLITICS
49393,Shinzo Abe To Become First Japanese Leader To ...,THE WORLDPOST
7057,Washington National Cathedral: John McCain's F...,POLITICS


In [17]:
# create mapping of labels to numeric values
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for idx, label in enumerate(labels)}
print(f"Label to ID mapping: {label2id}")
print(f"ID to Label mapping: {id2label}")

Label to ID mapping: {'U.S. NEWS': 0, 'COMEDY': 1, 'PARENTING': 2, 'WORLD NEWS': 3, 'CULTURE & ARTS': 4, 'TECH': 5, 'SPORTS': 6, 'ENTERTAINMENT': 7, 'POLITICS': 8, 'WEIRD NEWS': 9, 'ENVIRONMENT': 10, 'EDUCATION': 11, 'CRIME': 12, 'SCIENCE': 13, 'WELLNESS': 14, 'BUSINESS': 15, 'STYLE & BEAUTY': 16, 'FOOD & DRINK': 17, 'MEDIA': 18, 'QUEER VOICES': 19, 'HOME & LIVING': 20, 'WOMEN': 21, 'BLACK VOICES': 22, 'TRAVEL': 23, 'MONEY': 24, 'RELIGION': 25, 'LATINO VOICES': 26, 'IMPACT': 27, 'WEDDINGS': 28, 'COLLEGE': 29, 'PARENTS': 30, 'ARTS & CULTURE': 31, 'STYLE': 32, 'GREEN': 33, 'TASTE': 34, 'HEALTHY LIVING': 35, 'THE WORLDPOST': 36, 'GOOD NEWS': 37, 'WORLDPOST': 38, 'FIFTY': 39, 'ARTS': 40, 'DIVORCE': 41}
ID to Label mapping: {0: 'U.S. NEWS', 1: 'COMEDY', 2: 'PARENTING', 3: 'WORLD NEWS', 4: 'CULTURE & ARTS', 5: 'TECH', 6: 'SPORTS', 7: 'ENTERTAINMENT', 8: 'POLITICS', 9: 'WEIRD NEWS', 10: 'ENVIRONMENT', 11: 'EDUCATION', 12: 'CRIME', 13: 'SCIENCE', 14: 'WELLNESS', 15: 'BUSINESS', 16: 'STYLE & BE

In [20]:
# Map the labels in the dataset to their corresponding numeric values
def map_labels(example):
    example['labels'] = label2id[example['labels']]
    return example

label_mapped_dataset = dataset.map(map_labels)
# check some sample data after mapping
print(label_mapped_dataset['train'][17])


{'text': 'Maury Wills, Base-Stealing Shortstop For Dodgers, Dies At 89 Maury Wills, who helped the Los Angeles Dodgers win three World Series titles with his base-stealing prowess, has died.', 'labels': 6}
